# Chapter 1: From Pixels to Sequences — Vision Transformer

*Build a Multimodal Model from Scratch*

---

> **Prerequisites**: This book assumes you have read *Build a Large Language
> Model from Scratch* and are comfortable with Transformers, multi-head
> attention, residual connections, and layer normalization.  Those concepts
> are not re-explained here; we focus on what is *new* for the image modality.

---

## Chapter Goal

Build a **Vision Transformer (ViT)** from scratch inside this notebook.
ViT is the image encoder used by almost every modern multimodal model —
CLIP, LLaVA, Flamingo, GPT-4V all rely on some variant of it.

By the end of this chapter you will understand:
1. Why we cannot feed raw pixels into a Transformer.
2. How `PatchEmbedding` turns an image into a token sequence.
3. Why ViT uses *bidirectional* attention while GPT uses *causal* attention.
4. How the `[CLS]` token provides a global image representation.
5. That the entire ViT can be assembled from components you already know.

In [ ]:
import os, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.patches as mpatches

os.makedirs('figures', exist_ok=True)
torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {DEVICE}')

---
## 1.1  Why Images Need Special Treatment

You might wonder: Transformers handle arbitrary sequences — why not treat
each *pixel* as a token?

Let's do the arithmetic:

```
224 × 224 × 3 = 150,528  tokens  per image
```

GPT-2 processes sequences of up to 1,024 tokens.  With 150K tokens per
image, attention would require a matrix of size 150K × 150K = **22.5 billion
elements** — completely infeasible.

ViT's insight is deceptively simple:

> **Divide the image into non-overlapping 16×16-pixel patches.
> Treat each patch as a token.  Feed the sequence to a standard Transformer.**

For a 224×224 image with 16×16 patches that gives:
```
(224/16)² = 196  tokens  per image
```

196 tokens is totally manageable.  And each token is a compact 768-dimensional
embedding (rather than a raw 16×16×3 = 768-dimensional vector — same size,
much more expressive when learned!).

The idea was introduced in the 2020 paper
*"An Image is Worth 16×16 Words"*  — the title says it all.

In [ ]:
def draw_patch_splitting():
    import matplotlib.patheffects as pe
    DARK  = '#2C3E50'
    GREY  = '#BDC3C7'
    BLUE  = '#4A90D9'

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    fig.patch.set_facecolor('#F8F9FA')

    img_size   = 48
    patch_size = 16
    img = np.zeros((img_size, img_size, 3))
    img[:24, :24] = [0.9, 0.3, 0.3]
    img[:24, 24:] = [0.3, 0.8, 0.3]
    img[24:, :24] = [0.3, 0.4, 0.9]
    img[24:, 24:] = [0.95, 0.85, 0.2]
    np.random.seed(42)
    img = np.clip(img + np.random.rand(img_size, img_size, 3) * 0.12, 0, 1)

    axes[0].imshow(img)
    axes[0].set_title('(1) Original Image\n(48x48 pixels)', fontsize=11, fontweight='bold')
    axes[0].set_xticks([]); axes[0].set_yticks([])
    for s in axes[0].spines.values(): s.set_edgecolor(BLUE); s.set_linewidth(2)

    axes[1].imshow(img)
    n = img_size // patch_size
    for i in range(0, img_size+1, patch_size):
        axes[1].axhline(i-0.5, color='white', lw=2)
        axes[1].axvline(i-0.5, color='white', lw=2)
    for row in range(n):
        for col in range(n):
            axes[1].text(col*patch_size+patch_size//2, row*patch_size+patch_size//2,
                         f'P{row*n+col+1}', ha='center', va='center',
                         fontsize=10, color='white', fontweight='bold',
                         path_effects=[pe.withStroke(linewidth=2, foreground='black')])
    axes[1].set_title(f'(2) Split into {n*n} Patches\n(each {patch_size}x{patch_size} pixels)',
                      fontsize=11, fontweight='bold')
    axes[1].set_xticks([]); axes[1].set_yticks([])

    axes[2].set_xlim(0, n*n+1); axes[2].set_ylim(-0.5, 1.8)
    axes[2].axis('off')
    axes[2].set_title(f'(3) Flatten to Token Sequence\n({n*n} vectors)',
                      fontsize=11, fontweight='bold')
    for i in range(n*n):
        row, col = i//n, i%n
        patch_img = img[row*patch_size:(row+1)*patch_size, col*patch_size:(col+1)*patch_size]
        inset = axes[2].inset_axes([i/(n*n)+0.01, 0.35, 0.85/(n*n), 0.55])
        inset.imshow(patch_img); inset.set_xticks([]); inset.set_yticks([])
        for s in inset.spines.values(): s.set_edgecolor(DARK); s.set_linewidth(1.5)
        axes[2].text(i/(n*n)+0.5/(n*n)+0.01, 0.22, f't_{i+1}', ha='center',
                     fontsize=9, color=DARK, transform=axes[2].transAxes)

    plt.suptitle('Key Idea: Treat Image Patches Like Words',
                 fontsize=13, fontweight='bold', color=DARK, y=1.02)
    plt.tight_layout()
    plt.savefig('figures/ch01_patch_split.png', dpi=120, bbox_inches='tight')
    plt.show()

draw_patch_splitting()

---
## 1.2  Patch Embedding: From Scratch

### 1.2.1  Naive Implementation: Manual Loop

Let's start with the most transparent implementation so we understand
exactly what's happening:

In [ ]:
def naive_patch_embed(x, patch_size, embed_dim):
    """
    x: (B, C, H, W)
    Returns: (B, N_patches, embed_dim) after a linear projection per patch.
    """
    B, C, H, W = x.shape
    n_h = H // patch_size   # number of patches along height
    n_w = W // patch_size   # number of patches along width

    # Flatten each patch: C * patch_size * patch_size values
    patch_dim = C * patch_size * patch_size
    linear    = nn.Linear(patch_dim, embed_dim)  # one shared projection

    patches = []
    for row in range(n_h):
        for col in range(n_w):
            # Extract patch: (B, C, patch_size, patch_size)
            patch = x[:, :,
                      row*patch_size:(row+1)*patch_size,
                      col*patch_size:(col+1)*patch_size]
            flat = patch.flatten(1)          # (B, patch_dim)
            projected = linear(flat)          # (B, embed_dim)
            patches.append(projected)

    return torch.stack(patches, dim=1)       # (B, N_patches, embed_dim)

# Test
x = torch.randn(2, 3, 32, 32)
out = naive_patch_embed(x, patch_size=8, embed_dim=64)
print(f'Input shape : {x.shape}')
print(f'Output shape: {out.shape}   (2 images, {(32//8)**2} patches, 64-dim each)')

### 1.2.2  Elegant Implementation: Replace the Loop with Conv2d

The naive loop has two problems:
1. Python loops are slow — we want everything vectorized.
2. We need a linear projection after extracting each patch.

There is a beautiful trick: **a `Conv2d` with `kernel_size = stride = patch_size`
is mathematically equivalent to the loop above**.

**Why?**  A convolution slides a kernel over the input.  When `stride = kernel_size`
the kernels tile the image *without overlap* — each kernel covers exactly one
patch.  And a convolutional kernel IS a linear projection: it multiplies each
patch position by a weight matrix and sums.

```
           Loop                   Conv2d
   extract patch (H loop)   ┐
   extract patch (W loop)   ├  ≡  nn.Conv2d(C, D, kernel_size=P, stride=P)
   apply Linear projection  ┘
```

Let's verify this equivalence numerically:

In [ ]:
class PatchEmbeddingNaive(nn.Module):
    def __init__(self, patch_size, in_channels, embed_dim):
        super().__init__()
        self.patch_size = patch_size
        self.linear     = nn.Linear(in_channels * patch_size * patch_size, embed_dim)

    def forward(self, x):
        B, C, H, W = x.shape
        P = self.patch_size
        patches = []
        for r in range(H // P):
            for c in range(W // P):
                patch = x[:, :, r*P:(r+1)*P, c*P:(c+1)*P].flatten(1)
                patches.append(self.linear(patch))
        return torch.stack(patches, dim=1)


class PatchEmbeddingConv(nn.Module):
    def __init__(self, patch_size, in_channels, embed_dim):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)  # (B, N, D)


# Copy weights from naive to conv so they're identical
torch.manual_seed(0)
naive = PatchEmbeddingNaive(patch_size=8, in_channels=3, embed_dim=64)
conv  = PatchEmbeddingConv (patch_size=8, in_channels=3, embed_dim=64)

# Conv weight has shape (embed_dim, in_channels, P, P)
# Linear weight has shape (embed_dim, in_channels * P * P)
# They're the same weights, just reshaped
conv.proj.weight.data = naive.linear.weight.data.reshape(64, 3, 8, 8)
conv.proj.bias.data   = naive.linear.bias.data

x = torch.randn(2, 3, 32, 32)
out_naive = naive(x)
out_conv  = conv(x)

print(f'Naive output shape : {out_naive.shape}')
print(f'Conv  output shape : {out_conv.shape}')
print(f'Outputs identical  : {torch.allclose(out_naive, out_conv, atol=1e-5)}')
print()
print('The Conv2d trick is not an approximation — it is exactly the same computation.')

### 1.2.3  The Final PatchEmbedding Class

With that understood, here is the clean, production-quality implementation:

In [ ]:
class PatchEmbedding(nn.Module):
    """
    Splits an image into non-overlapping patches and projects each to embed_dim.

    Conv2d with kernel_size=stride=patch_size performs both operations at once:
    - Extracts patches (non-overlapping when stride = kernel_size)
    - Projects each patch to embed_dim (convolutional weights = linear projection)
    """
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        assert img_size % patch_size == 0, 'img_size must be divisible by patch_size'
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x):          # x: (B, C, H, W)
        x = self.proj(x)           # (B, embed_dim, H/P, W/P)
        x = x.flatten(2)           # (B, embed_dim, N_patches)
        x = x.transpose(1, 2)      # (B, N_patches, embed_dim)
        return x


pe = PatchEmbedding(img_size=32, patch_size=8, in_channels=3, embed_dim=128)
x  = torch.randn(4, 3, 32, 32)    # batch of 4 images, 32x32
out = pe(x)
print(f'Input : {x.shape}  →  Output: {out.shape}')
print(f'Number of patches: {pe.n_patches}  (= (32/8)^2 = 16)')

---
## 1.3  Positional Embedding

### Why Do Images Need Position Information?

Transformers are **permutation invariant** — shuffling the input tokens
produces the same attention outputs (just in a different order).  That is
fine for some tasks, but catastrophic for images: the patch in the top-left
corner has completely different meaning from the patch in the bottom-right corner.

Let's demonstrate this concretely:

In [ ]:
def show_shuffle_effect():
    """Show that shuffling patches destroys the image."""
    torch.manual_seed(7)
    img = torch.zeros(1, 3, 32, 32)
    # Make a recognizable pattern: red in top-left, blue in bottom-right
    img[0, 0, :16, :16] = 0.9   # red quadrant
    img[0, 2, 16:, 16:] = 0.9   # blue quadrant

    pe_layer = PatchEmbedding(img_size=32, patch_size=8, in_channels=3, embed_dim=3)
    # Use identity-like weights so we can visualize the patches
    nn.init.eye_(pe_layer.proj.weight.view(3, -1)[:3, :3])
    pe_layer.proj.weight.data[:, :, :, :] = 0
    for i in range(3):
        pe_layer.proj.weight.data[i, i, 0, 0] = 1.0
    pe_layer.proj.bias.data.zero_()

    with torch.no_grad():
        tokens = pe_layer(img)          # (1, 16, 3)

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

    def tokens_to_img(tok):
        # Reconstruct a rough image from the 3-channel patch tokens
        # (very approximate, just for illustration)
        n = int(tok.shape[1] ** 0.5)
        grid = tok.reshape(n, n, 3).clamp(0, 1).numpy()
        return grid

    # Original
    orig = img[0].permute(1, 2, 0).clamp(0, 1).numpy()
    axes[0].imshow(orig)
    axes[0].set_title('Original Image', fontweight='bold')
    axes[0].axis('off')

    # Patch grid (original order)
    n_patches = tokens.shape[1]
    n_side    = int(n_patches ** 0.5)
    grid = img[0].permute(1, 2, 0).numpy()
    axes[1].imshow(orig)
    for r in range(n_side + 1):
        axes[1].axhline(r * 8 - 0.5, color='white', lw=1.5)
        axes[1].axvline(r * 8 - 0.5, color='white', lw=1.5)
    axes[1].set_title('Patches (original order)', fontweight='bold')
    axes[1].axis('off')

    # Shuffle the patches
    perm = torch.randperm(n_patches)
    shuffled_tokens = tokens[:, perm, :]
    shuffled_img = torch.zeros_like(img)
    for new_pos, old_pos in enumerate(perm.tolist()):
        old_r, old_c = old_pos // n_side, old_pos % n_side
        new_r, new_c = new_pos // n_side, new_pos % n_side
        shuffled_img[0, :, new_r*8:(new_r+1)*8, new_c*8:(new_c+1)*8] = \
            img[0, :, old_r*8:(old_r+1)*8, old_c*8:(old_c+1)*8]

    shuffled_np = shuffled_img[0].permute(1, 2, 0).clamp(0, 1).numpy()
    axes[2].imshow(shuffled_np)
    axes[2].set_title('After Shuffling Patches\n(model sees this without pos. embed!)',
                       fontweight='bold', color='red')
    axes[2].axis('off')

    plt.suptitle('Motivation for Positional Embeddings', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/ch01_shuffle.png', dpi=110, bbox_inches='tight')
    plt.show()

show_shuffle_effect()

ViT uses **learnable positional embeddings** (as opposed to the fixed
sinusoidal embeddings in the original Transformer or GPT).  On fixed-resolution
images, learned embeddings perform at least as well and are simpler.

The embedding is a parameter tensor of shape `(1, n_patches + 1, embed_dim)`.
The `+ 1` is for the `[CLS]` token we add next.  It is initialized near zero
and trained via gradient descent.

In [ ]:
class PositionalEmbedding(nn.Module):
    """
    Learnable positional embedding added to patch tokens + CLS token.
    Shape: (1, n_patches + 1, embed_dim)  — broadcasted over batch dimension.
    """
    def __init__(self, n_patches, embed_dim):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):   # x: (B, n_patches+1, embed_dim)
        return x + self.pos_embed

print('PositionalEmbedding defined.')

---
## 1.4  The [CLS] Token — Global Image Representation

The Transformer outputs a vector for each input token.  We typically need a
*single* vector to represent the whole image (e.g., for classification or for
feeding into a larger model).  Two options:

| Method | Pros | Cons |
|--------|------|------|
| Average all patch outputs | Simple | Weights every patch equally; no learned aggregation |
| Prepend a learnable [CLS] token | Learns to aggregate via attention | Adds one token to the sequence |

ViT (and BERT) uses the **[CLS] token** approach.  This special token starts
as a random learnable vector.  After passing through all Transformer layers
it has attended to every patch and learned to summarize the entire image.

```
Input:  [CLS] | P_1 | P_2 | ... | P_N
Output: [CLS]'| P_1'| P_2'| ... | P_N'
                ↑
        This CLS output = global image representation
```

The CLS token is initialized with `trunc_normal_(std=0.02)` — small random
values that allow the network to set it to whatever it needs through training.

---
## 1.5  ViT Attention — The One Difference From GPT

If you finished the original book, you fully understand multi-head attention.
ViT and GPT use the same attention mechanism with one critical difference:

| | GPT (language model) | ViT (image encoder) |
|--|--|--|
| Attention type | **Causal** | **Bidirectional** |
| Mask | Upper-triangle masked: cannot attend to future tokens | No mask: every patch attends to every other patch |
| Reason | Text is generated left-to-right; future tokens must be unseen | Images have no temporal order; every patch can inform every other |

The figure below shows the two attention patterns side by side:

In [ ]:
def draw_attention_masks():
    DARK  = '#2C3E50'
    RED   = '#D9534F'
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    fig.patch.set_facecolor('#F8F9FA')
    T = 6
    labels = [f'P{i+1}' for i in range(T)]

    for ax, title, is_causal, cmap, sub in [
        (axes[0], 'ViT  --  Bidirectional Attention', False, 'Blues',
         'Every patch attends to every other patch'),
        (axes[1], 'GPT  --  Causal Attention', True, 'Oranges',
         'Each token only sees past tokens'),
    ]:
        mask = np.tril(np.ones((T, T))) if is_causal else np.ones((T, T))
        ax.imshow(mask, cmap=cmap, vmin=0, vmax=1.3, aspect='equal',
                  interpolation='nearest')
        ax.set_xticks(range(T)); ax.set_yticks(range(T))
        ax.set_xticklabels(labels, fontsize=9)
        ax.set_yticklabels(labels, fontsize=9)
        ax.set_xlabel('Key (attends TO)', fontsize=10)
        ax.set_ylabel('Query (attending FROM)', fontsize=10)
        ax.set_title(f'{title}\n{sub}', fontsize=11, fontweight='bold', pad=8)
        for i in range(T):
            for j in range(T):
                v = mask[i, j]
                ax.text(j, i, 'OK' if v > 0 else 'X', ha='center', va='center',
                        fontsize=11, color=DARK if v > 0 else RED, fontweight='bold')
            rect = plt.Rectangle((i-0.5, i-0.5), 1, 1,
                                  fill=False, edgecolor=RED, lw=2.5)
            ax.add_patch(rect)

    axes[1].text(3.5, -1.2, '<-- Blocked (future tokens)',
                 ha='center', fontsize=9, color=RED, style='italic')
    plt.suptitle('Bidirectional vs Causal Attention -- The Key Difference',
                 fontsize=13, fontweight='bold', color=DARK)
    plt.tight_layout()
    plt.savefig('figures/ch01_attn_masks.png', dpi=120, bbox_inches='tight')
    plt.show()

draw_attention_masks()

In code, dropping the causal mask requires removing exactly one line:

```python
# GPT (causal attention) — add this mask:
mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
scores.masked_fill_(mask, float('-inf'))

# ViT (bidirectional) — simply do NOT add the mask.
# That's it.
```

This is the most important difference to internalize before moving on.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """
    Multi-head self-attention WITHOUT a causal mask.
    This is the bidirectional version used by ViT.
    """
    def __init__(self, embed_dim, n_heads, dropout=0.0):
        super().__init__()
        assert embed_dim % n_heads == 0
        self.n_heads  = n_heads
        self.head_dim = embed_dim // n_heads
        self.scale    = self.head_dim ** -0.5
        self.qkv      = nn.Linear(embed_dim, 3 * embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.drop     = nn.Dropout(dropout)

    def forward(self, x):
        B, N, C = x.shape
        # Project to Q, K, V
        qkv = self.qkv(x).reshape(B, N, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        # Attention scores — NO masked_fill here (bidirectional)
        attn = F.softmax((q @ k.transpose(-2, -1)) * self.scale, dim=-1)
        attn = self.drop(attn)
        x    = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.out_proj(x)

print('MultiHeadSelfAttention defined  (bidirectional, no causal mask).')

---
## 1.6  Assembling the Complete ViT

All components are now defined.  The ViT Transformer block is identical to
GPT's block (Pre-LayerNorm formulation):

```
x = x + Attention(LayerNorm(x))   # residual + attention
x = x + FFN(LayerNorm(x))         # residual + feed-forward
```

The only change: the attention has no causal mask.

In [ ]:
class ViTBlock(nn.Module):
    def __init__(self, embed_dim, n_heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = MultiHeadSelfAttention(embed_dim, n_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        hidden     = int(embed_dim * mlp_ratio)
        self.ffn   = nn.Sequential(
            nn.Linear(embed_dim, hidden), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, embed_dim), nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))   # Pre-LN
        x = x + self.ffn(self.norm2(x))
        return x


class VisionTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3,
                 embed_dim=768, depth=12, n_heads=12, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        n_patches = self.patch_embed.n_patches

        # Learnable [CLS] token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # Positional embedding covers CLS + all patch tokens
        self.pos_embed = PositionalEmbedding(n_patches, embed_dim)

        self.blocks  = nn.ModuleList([ViTBlock(embed_dim, n_heads, dropout=dropout)
                                       for _ in range(depth)])
        self.norm    = nn.LayerNorm(embed_dim)

        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x, return_all_tokens=False):
        B = x.shape[0]
        # 1. Extract patch tokens
        tokens = self.patch_embed(x)                  # (B, N, D)

        # 2. Prepend CLS token to the sequence
        cls    = self.cls_token.expand(B, -1, -1)     # (B, 1, D)
        tokens = torch.cat([cls, tokens], dim=1)      # (B, N+1, D)

        # 3. Add positional embedding
        tokens = self.pos_embed(tokens)

        # 4. Pass through Transformer blocks
        for blk in self.blocks:
            tokens = blk(tokens)
        tokens = self.norm(tokens)

        # CLS output = global image representation
        cls_out = tokens[:, 0, :]       # (B, D)
        if return_all_tokens:
            return {'cls': cls_out, 'tokens': tokens[:, 1:, :]}
        return cls_out

# Quick shape test
vit  = VisionTransformer(img_size=32, patch_size=8, embed_dim=128, depth=4, n_heads=4)
imgs = torch.randn(2, 3, 32, 32)
out  = vit(imgs)
print(f'Input : {imgs.shape}')
print(f'Output (CLS): {out.shape}')
total = sum(p.numel() for p in vit.parameters())
print(f'Total parameters: {total:,}')

---
## 1.7  Training Demo: 4-Class Image Classification

We build a synthetic dataset with four visually distinct classes to verify
that our ViT implementation actually learns.

| Class | Pattern | Label |
|-------|---------|-------|
| 0 | Solid red | `red` |
| 1 | Solid blue | `blue` |
| 2 | Vertical red/blue stripes | `v-stripes` |
| 3 | Horizontal red/blue stripes | `h-stripes` |

The patterns are simple enough that the model should converge to near-perfect
accuracy in a few dozen epochs on a CPU.

In [ ]:
class SyntheticImageDataset(Dataset):
    """4-class geometric image dataset for ViT training validation."""

    def __init__(self, n_per_class=200, img_size=32, seed=42):
        torch.manual_seed(seed)
        self.data = []
        for label in range(4):
            for _ in range(n_per_class):
                img = torch.zeros(3, img_size, img_size)
                if label == 0:   # solid red
                    img[0] = 0.9
                elif label == 1: # solid blue
                    img[2] = 0.9
                elif label == 2: # vertical stripes
                    for c in range(img_size):
                        img[0 if c % 8 < 4 else 2, :, c] = 0.9
                else:            # horizontal stripes
                    for r in range(img_size):
                        img[0 if r % 8 < 4 else 2, r, :] = 0.9
                img += torch.randn_like(img) * 0.05  # small noise
                self.data.append((img, label))

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]


train_set = SyntheticImageDataset(n_per_class=200)
test_set  = SyntheticImageDataset(n_per_class=50, seed=99)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=32, shuffle=False)
print(f'Training set: {len(train_set)}  Test set: {len(test_set)}')

In [ ]:
class ViTClassifier(nn.Module):
    def __init__(self, n_classes, **vit_kwargs):
        super().__init__()
        self.vit  = VisionTransformer(**vit_kwargs)
        embed_dim = vit_kwargs.get('embed_dim', 768)
        self.head = nn.Linear(embed_dim, n_classes)

    def forward(self, x):
        cls = self.vit(x)          # (B, embed_dim)
        return self.head(cls)      # (B, n_classes)


model     = ViTClassifier(n_classes=4, img_size=32, patch_size=8,
                          embed_dim=128, depth=4, n_heads=4, dropout=0.1)
model     = model.to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            preds = model(imgs).argmax(1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return correct / total

train_losses, test_accs = [], []
EPOCHS = 30
print(f'Training for {EPOCHS} epochs ...')
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        loss = F.cross_entropy(model(imgs), labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    acc      = evaluate(model, test_loader)
    train_losses.append(avg_loss)
    test_accs.append(acc)
    if (epoch + 1) % 5 == 0:
        print(f'  Epoch {epoch+1:3d}/{EPOCHS}  loss={avg_loss:.4f}  test_acc={acc:.2%}')

print(f'\nFinal test accuracy: {test_accs[-1]:.2%}')

In [ ]:
def plot_training_results(losses, accs):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

    epochs = range(1, len(losses) + 1)
    ax1.plot(epochs, losses, color='#3b82f6', lw=2.5)
    ax1.axhline(math.log(4), color='red', linestyle='--', alpha=0.7,
                label=f'Random baseline: ln(4)={math.log(4):.2f}')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-Entropy Loss')
    ax1.set_title('Training Loss', fontweight='bold')
    ax1.legend(); ax1.grid(axis='y', alpha=0.3)

    ax2.plot(epochs, [a*100 for a in accs], color='#22c55e', lw=2.5)
    ax2.axhline(25, color='red', linestyle='--', alpha=0.7, label='Random (25%)')
    ax2.axhline(100, color='grey', linestyle=':', alpha=0.4)
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Test Accuracy (%)')
    ax2.set_title('Test Accuracy', fontweight='bold')
    ax2.set_ylim(0, 105); ax2.legend(); ax2.grid(axis='y', alpha=0.3)

    plt.suptitle('ViT Training on Synthetic 4-Class Dataset',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/ch01_training.png', dpi=120, bbox_inches='tight')
    plt.show()

plot_training_results(train_losses, test_accs)

---
## 1.8  Chapter Summary

| Component | Role | Key Detail |
|-----------|------|------------|
| `PatchEmbedding` | Image → token sequence | Conv2d with stride=kernel_size=patch_size; mathematically identical to loop + Linear |
| `PositionalEmbedding` | Inject spatial position | Learnable parameter `(1, N+1, D)`; initialized near zero |
| `[CLS] token` | Global image representation | Learnable vector prepended to patches; trained by gradient descent |
| `MultiHeadSelfAttention` | Patch-to-patch relationships | **No causal mask** — the only architectural difference from GPT |
| `ViTBlock` | Transformer layer | Pre-LN: `x = x + Attn(Norm(x))`; `x = x + FFN(Norm(x))` |
| `VisionTransformer` | Full image encoder | Returns CLS output as global representation |

**The single most important thing to remember:**

> ViT and GPT share almost identical code.
> ViT is **bidirectional** (no mask); GPT is **causal** (upper-triangle mask).
> This one difference has profound consequences for how information flows.

**Next — Chapter 2: CLIP**

ViT gives us rich image representations, but they live in "image space" —
a very different coordinate system from language model embeddings.
Chapter 2 trains a dual encoder with contrastive learning to map both
images and text into a *shared* semantic space.